In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

In [20]:
# Đường dẫn đến dữ liệu gốc
file_path = Path("../data/online_retail_II.xlsx")

# Đọc hai sheet
df_2009 = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010"
)

df_2010 = pd.read_excel(
    file_path,
    sheet_name="Year 2010-2011"
)

# Thêm cột xác định nguồn dữ liệu
df_2009["SourceSheet"] = "Year 2009-2010"
df_2010["SourceSheet"] = "Year 2010-2011"

# Kết hợp hai sheet
df = pd.concat(
    [df_2009, df_2010],
    ignore_index=True
)

print("Kích thước dữ liệu:", df.shape)

Kích thước dữ liệu: (1067371, 9)


In [21]:
# Lưu bản dữ liệu ban đầu
df_raw = df.copy()

# Tạo bản dữ liệu dùng để làm sạch
df_clean = df.copy()

print("Dữ liệu gốc:", df_raw.shape)
print("Dữ liệu làm sạch:", df_clean.shape)

Dữ liệu gốc: (1067371, 9)
Dữ liệu làm sạch: (1067371, 9)


In [22]:
# 1. Chuẩn hóa Invoice
df_clean["Invoice"] = df_clean["Invoice"].astype("string")

# 2. Chuẩn hóa Customer ID
customer_id = pd.to_numeric(df_clean["Customer ID"], errors="coerce")

# Kiểm tra Customer ID có phải số nguyên không
valid_ids = customer_id.dropna()

if not (valid_ids % 1 == 0).all():
    raise ValueError("Customer ID có giá trị không phải số nguyên")

df_clean["Customer ID"] = customer_id.astype("Int64")

# 3. Chuẩn hóa văn bản
text_cols = df_clean.select_dtypes(include=["object", "string"]).columns

for col in text_cols:
    df_clean[col] = df_clean[col].astype("string").str.strip()

# 4. Kiểm tra kết quả
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  string        
 1   StockCode    1067371 non-null  string        
 2   Description  1062989 non-null  string        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   Int64         
 7   Country      1067371 non-null  string        
 8   SourceSheet  1067371 non-null  string        
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), string(5)
memory usage: 74.3 MB


In [23]:
# Xem 5 dòng đầu
display(df_clean.head())

# Kiểm tra kích thước
print("Kích thước:", df_clean.shape)

# Kiểm tra kiểu dữ liệu
df_clean.info()

# Kiểm tra giá trị thiếu
display(df_clean.isnull().sum())

# Kiểm tra số dòng trùng lặp
print(
    "Số dòng trùng lặp:",
    df_clean.duplicated().sum()
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Year 2009-2010


Kích thước: (1067371, 9)
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  string        
 1   StockCode    1067371 non-null  string        
 2   Description  1062989 non-null  string        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   Int64         
 7   Country      1067371 non-null  string        
 8   SourceSheet  1067371 non-null  string        
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), string(5)
memory usage: 74.3 MB


Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
SourceSheet         0
dtype: int64

Số dòng trùng lặp: 12133


In [24]:
# Đếm số dòng trùng trước khi xử lý
duplicates_before = df_clean.duplicated().sum()

print("Số dòng trùng trước khi xử lý:", duplicates_before)

# Loại bỏ các dòng trùng hoàn toàn
df_clean = df_clean.drop_duplicates().copy()

# Kiểm tra sau khi xử lý
duplicates_after = df_clean.duplicated().sum()

print("Số dòng trùng sau khi xử lý:", duplicates_after)
print("Kích thước dữ liệu sau xử lý:", df_clean.shape)

# Số dòng đã loại bỏ
print("Số dòng đã loại bỏ:", len(df_raw) - len(df_clean))

Số dòng trùng trước khi xử lý: 12133
Số dòng trùng sau khi xử lý: 0
Kích thước dữ liệu sau xử lý: (1055238, 9)
Số dòng đã loại bỏ: 12133


In [25]:
# Kiểm tra số lượng giá trị thiếu
missing_values = df_clean.isnull().sum()

# Tính tỷ lệ phần trăm giá trị thiếu
missing_percent = (
    df_clean.isnull().sum() / len(df_clean) * 100
)

# Tạo bảng tổng hợp
missing_table = pd.DataFrame({
    "Missing Values": missing_values,
    "Percentage (%)": missing_percent
})

display(missing_table)

,Missing Values,Percentage (%)
Invoice,0,0.000000
StockCode,0,0.000000
Description,4382,0.415262
Quantity,0,0.000000
InvoiceDate,0,0.000000
Price,0,0.000000
Customer ID,242870,23.015661
Country,0,0.000000
SourceSheet,0,0.000000


In [26]:
# Đếm số giá trị thiếu trước khi xử lý
print(
    "Description thiếu trước xử lý:",
    df_clean["Description"].isnull().sum()
)

# Thay giá trị thiếu
df_clean["Description"] = (
    df_clean["Description"].fillna("Unknown Product")
)

# Kiểm tra sau xử lý
print(
    "Description thiếu sau xử lý:",
    df_clean["Description"].isnull().sum()
)
print(
    "Customer ID còn thiếu:",
    df_clean["Customer ID"].isnull().sum()
)

print(
    "Tỷ lệ Customer ID còn thiếu:",
    df_clean["Customer ID"].isnull().mean() * 100,
    "%"
)

Description thiếu trước xử lý: 4382
Description thiếu sau xử lý: 0
Customer ID còn thiếu: 242870
Tỷ lệ Customer ID còn thiếu: 23.015660921991056 %


In [27]:
# Kiểm tra giá trị thiếu sau xử lý
display(df_clean.isnull().sum())

# Kiểm tra kích thước dữ liệu
print("Kích thước dữ liệu:", df_clean.shape)

Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    242870
Country             0
SourceSheet         0
dtype: int64

Kích thước dữ liệu: (1055238, 9)


In [28]:
# Kiểm tra hóa đơn hủy
df_clean["IsCancelled"] = (
    df_clean["Invoice"].astype(str).str.startswith("C")
)

print("Số dòng giao dịch hủy:",
      df_clean["IsCancelled"].sum())

print("Số dòng giao dịch bình thường:",
      (~df_clean["IsCancelled"]).sum())

# Tỷ lệ giao dịch hủy
print("Tỷ lệ giao dịch hủy:",
      df_clean["IsCancelled"].mean() * 100, "%")

Số dòng giao dịch hủy: 19433
Số dòng giao dịch bình thường: 1035805
Tỷ lệ giao dịch hủy: 1.8415750759544294 %


In [29]:
# Thống kê số lượng và giá trị giao dịch hủy
cancelled_data = df_clean[df_clean["IsCancelled"] == True]

print("Số dòng giao dịch hủy:", len(cancelled_data))

print("Tổng Quantity giao dịch hủy:",
      cancelled_data["Quantity"].sum())

print("Tổng Price giao dịch hủy:",
      cancelled_data["Price"].sum())

display(cancelled_data.head(10))

Số dòng giao dịch hủy: 19433
Tổng Quantity giao dịch hủy: -488848
Tổng Price giao dịch hủy: 874471.05


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancelled
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia,Year 2009-2010,True
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia,Year 2009-2010,True
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia,Year 2009-2010,True
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia,Year 2009-2010,True
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia,Year 2009-2010,True
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321,Australia,Year 2009-2010,True
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321,Australia,Year 2009-2010,True
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321,Australia,Year 2009-2010,True
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321,Australia,Year 2009-2010,True
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592,United Kingdom,Year 2009-2010,True


In [30]:
# Giao dịch có Quantity âm nhưng không phải hóa đơn hủy
negative_quantity = df_clean[
    (df_clean["Quantity"] < 0) &
    (df_clean["IsCancelled"] == False)
]

print("Số dòng Quantity âm không thuộc hóa đơn hủy:",
      len(negative_quantity))

print("Tổng Quantity:",
      negative_quantity["Quantity"].sum())

print("Số dòng có Price = 0:",
      (negative_quantity["Price"] == 0).sum())

print("Số dòng thiếu Customer ID:",
      negative_quantity["Customer ID"].isnull().sum())

# Xem một số dòng dữ liệu
display(negative_quantity.head(10))

Số dòng Quantity âm không thuộc hóa đơn hủy: 3457
Tổng Quantity: -573085
Số dòng có Price = 0: 3457
Số dòng thiếu Customer ID: 3457


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancelled
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,Year 2009-2010,False
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,Year 2009-2010,False
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,<NA>,United Kingdom,Year 2009-2010,False
470,489521,21646,Unknown Product,-50,2009-12-01 11:44:00,0.0,<NA>,United Kingdom,Year 2009-2010,False
3114,489655,20683,Unknown Product,-44,2009-12-01 17:26:00,0.0,<NA>,United Kingdom,Year 2009-2010,False
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,<NA>,United Kingdom,Year 2009-2010,False
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,<NA>,United Kingdom,Year 2009-2010,False
4296,489806,18010,Unknown Product,-770,2009-12-02 12:42:00,0.0,<NA>,United Kingdom,Year 2009-2010,False
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,<NA>,United Kingdom,Year 2009-2010,False
4566,489821,85049G,Unknown Product,-240,2009-12-02 13:25:00,0.0,<NA>,United Kingdom,Year 2009-2010,False


In [31]:
# Đánh dấu các giao dịch có Quantity âm nhưng không phải hóa đơn hủy
df_clean["IsSpecialAdjustment"] = (
    (df_clean["Quantity"] < 0) &
    (df_clean["IsCancelled"] == False)
)

print("Số dòng giao dịch đặc biệt:",
      df_clean["IsSpecialAdjustment"].sum())

print("Số dòng không thuộc giao dịch đặc biệt:",
      (~df_clean["IsSpecialAdjustment"]).sum())


Số dòng giao dịch đặc biệt: 3457
Số dòng không thuộc giao dịch đặc biệt: 1051781


In [32]:
print("Số dòng Price âm:",
      (df_clean["Price"] < 0).sum())

print("Số dòng Price bằng 0:",
      (df_clean["Price"] == 0).sum())

print("Số dòng Price dương:",
      (df_clean["Price"] > 0).sum())

# Xem các dòng có Price âm
display(df_clean[df_clean["Price"] < 0].head(10))

Số dòng Price âm: 5
Số dòng Price bằng 0: 6191
Số dòng Price dương: 1049042


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancelled,IsSpecialAdjustment
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,<NA>,United Kingdom,Year 2009-2010,False,False
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,<NA>,United Kingdom,Year 2009-2010,False,False
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,<NA>,United Kingdom,Year 2009-2010,False,False
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,<NA>,United Kingdom,Year 2010-2011,False,False
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,<NA>,United Kingdom,Year 2010-2011,False,False


In [33]:
# Đánh dấu các dòng có Price âm
df_clean["IsNegativePrice"] = df_clean["Price"] < 0

print("Số dòng Price âm:",
      df_clean["IsNegativePrice"].sum())

print("Số dòng Price không âm:",
      (~df_clean["IsNegativePrice"]).sum())


Số dòng Price âm: 5
Số dòng Price không âm: 1055233


In [34]:
print("TỔNG HỢP DỮ LIỆU SAU LÀM SẠCH")

print("Tổng số dòng:", len(df_clean))
print("Giao dịch hủy:", df_clean["IsCancelled"].sum())
print("Giao dịch điều chỉnh đặc biệt:",
      df_clean["IsSpecialAdjustment"].sum())
print("Giao dịch có Price âm:",
      df_clean["IsNegativePrice"].sum())
print("Customer ID bị thiếu:",
      df_clean["Customer ID"].isnull().sum())
print("Description bị thiếu:",
      df_clean["Description"].isnull().sum())
print("Dòng trùng lặp:", df_clean.duplicated().sum())

TỔNG HỢP DỮ LIỆU SAU LÀM SẠCH
Tổng số dòng: 1055238
Giao dịch hủy: 19433
Giao dịch điều chỉnh đặc biệt: 3457
Giao dịch có Price âm: 5
Customer ID bị thiếu: 242870
Description bị thiếu: 0
Dòng trùng lặp: 0


In [35]:
# Tạo dữ liệu chỉ gồm giao dịch bán hàng thông thường
df_sales = df_clean[
    (df_clean["IsCancelled"] == False) &
    (df_clean["IsSpecialAdjustment"] == False) &
    (df_clean["Price"] > 0)
].copy()

print("Kích thước dữ liệu bán hàng:", df_sales.shape)

print("Số giao dịch hủy:",
      df_sales["IsCancelled"].sum())

print("Số giao dịch đặc biệt:",
      df_sales["IsSpecialAdjustment"].sum())

print("Số dòng Price không dương:",
      (df_sales["Price"] <= 0).sum())

display(df_sales.head())


Kích thước dữ liệu bán hàng: (1029609, 12)
Số giao dịch hủy: 0
Số giao dịch đặc biệt: 0
Số dòng Price không dương: 0


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancelled,IsSpecialAdjustment,IsNegativePrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Year 2009-2010,False,False,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,False,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,False,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Year 2009-2010,False,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Year 2009-2010,False,False,False


In [ ]:
# Tạo cột doanh thu
df_sales["Revenue"] = df_sales["Quantity"] * df_sales["Price"]

# Kiểm tra kết quả
display(
    df_sales[
        ["Invoice", "StockCode", "Quantity", "Price", "Revenue"]
    ].head(10)
)

print("Tổng doanh thu:", df_sales["Revenue"].sum())
print("Doanh thu nhỏ nhất:", df_sales["Revenue"].min())
print("Doanh thu lớn nhất:", df_sales["Revenue"].max())


,Invoice,StockCode,Quantity,Price,Revenue
0,489434,85048,12,6.95,83.4
1,489434,79323P,12,6.75,81.0
2,489434,79323W,12,6.75,81.0
3,489434,22041,48,2.10,100.8
4,489434,21232,24,1.25,30.0
5,489434,22064,24,1.65,39.6
6,489434,21871,24,1.25,30.0
7,489434,21523,10,5.95,59.5
8,489435,22350,12,2.55,30.6
9,489435,22349,12,3.75,45.0


Tổng doanh thu: 20913891.468000002
Doanh thu nhỏ nhất: 0.001
Doanh thu lớn nhất: 168469.6


In [40]:
# Chuẩn hóa kiểu dữ liệu cho df_sales
df_sales["Invoice"] = df_sales["Invoice"].astype("string")

customer_id = pd.to_numeric(
    df_sales["Customer ID"], errors="coerce"
)

df_sales["Customer ID"] = customer_id.astype("Int64")

print(df_sales.dtypes)

Invoice                        string
StockCode                      string
Description                    string
Quantity                        int64
InvoiceDate            datetime64[us]
Price                         float64
Customer ID                     Int64
Country                        string
SourceSheet                    string
IsCancelled                      bool
IsSpecialAdjustment              bool
IsNegativePrice                  bool
Revenue                       float64
dtype: object


In [37]:
# Sắp xếp doanh thu từ cao xuống thấp
top_revenue = df_sales.sort_values(
    by="Revenue",
    ascending=False
)

# Hiển thị 10 giao dịch có doanh thu cao nhất
display(
    top_revenue[
        ["Invoice", "StockCode", "Description",
         "Quantity", "Price", "Revenue"]
    ].head(10)
)

,Invoice,StockCode,Description,Quantity,Price,Revenue
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60
748132,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,649.50,38970.00
241827,512771,M,Manual,1,25111.09,25111.09
432176,530715,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,1.69,15818.40
517955,537632,AMAZONFEE,AMAZON FEE,1,13541.33,13541.33
540478,537632,AMAZONFEE,AMAZON FEE,1,13541.33,13541.33
825443,A563185,B,Adjust bad debt,1,11062.06,11062.06
135013,502263,M,Manual,1,10953.50,10953.50
135015,502265,M,Manual,1,10953.50,10953.50


In [45]:
# Lọc các dòng có Quantity âm
negative_quantity = df_clean[df_clean["Quantity"] < 0]

print("Số dòng Quantity âm:", len(negative_quantity))

# Hiển thị 20 dòng đầu tiên
display(negative_quantity.head(20))

Số dòng Quantity âm: 22889


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancelled,IsSpecialAdjustment,IsNegativePrice
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia,Year 2009-2010,True,False,False
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia,Year 2009-2010,True,False,False
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia,Year 2009-2010,True,False,False
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia,Year 2009-2010,True,False,False
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia,Year 2009-2010,True,False,False
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321,Australia,Year 2009-2010,True,False,False
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321,Australia,Year 2009-2010,True,False,False
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321,Australia,Year 2009-2010,True,False,False
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321,Australia,Year 2009-2010,True,False,False
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592,United Kingdom,Year 2009-2010,True,False,False


In [46]:
# Thống kê theo mô tả sản phẩm
print("Các loại giao dịch Quantity âm:")
display(
    negative_quantity["Description"]
    .value_counts()
    .head(15)
)

Các loại giao dịch Quantity âm:


Description
Unknown Product                       2689
Manual                                 536
REGENCY CAKESTAND 3 TIER               346
POSTAGE                                229
BAKING SET 9 PIECE RETROSPOT           211
STRAWBERRY CERAMIC TRINKET BOX         182
Discount                               172
WHITE HANGING HEART T-LIGHT HOLDER     135
WHITE CHERRY LIGHTS                    126
check                                  123
RED RETROSPOT CAKE STAND               111
SAMPLES                                100
JAM MAKING SET WITH JARS                90
WOOD 2 DRAWER CABINET WHITE FINISH      86
PINK CHERRY LIGHTS                      86
Name: count, dtype: Int64

In [47]:
print("Invoice bắt đầu bằng C:")
print(
    negative_quantity["Invoice"]
    .str.startswith("C", na=False)
    .value_counts()
)

Invoice bắt đầu bằng C:
Invoice
True     19432
False     3457
Name: count, dtype: Int64


In [51]:
# Lọc Quantity âm nhưng không phải hóa đơn hủy
negative_quantity_not_cancelled = df_clean[
    (df_clean["Quantity"] < 0) &
    (~df_clean["IsCancelled"])
]

print("Số dòng:", len(negative_quantity_not_cancelled))

# Xem 20 dòng đầu tiên
display(negative_quantity_not_cancelled.head(20))


Số dòng: 3457


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancelled,IsSpecialAdjustment,IsNegativePrice
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,True,False
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,True,False
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,True,False
470,489521,21646,Unknown Product,-50,2009-12-01 11:44:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,True,False
3114,489655,20683,Unknown Product,-44,2009-12-01 17:26:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,True,False
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,True,False
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,True,False
4296,489806,18010,Unknown Product,-770,2009-12-02 12:42:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,True,False
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,True,False
4566,489821,85049G,Unknown Product,-240,2009-12-02 13:25:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,True,False


In [52]:
display(
    negative_quantity_not_cancelled["Description"]
    .value_counts()
    .head(20)
)

Description
Unknown Product           2689
check                      123
damages                     84
?                           83
damaged                     78
missing                     27
sold as set on dotcom       20
Damaged                     17
smashed                      9
thrown away                  9
Unsaleable, destroyed.       9
dotcom                       8
damages?                     7
??                           7
crushed                      6
given away                   6
MIA                          5
Damages                      5
counted                      5
checked                      5
Name: count, dtype: Int64

In [53]:
display(
    negative_quantity_not_cancelled[
        ["Invoice", "StockCode", "Description",
         "Quantity", "Price", "Customer ID"]
    ].head(30)
)

,Invoice,StockCode,Description,Quantity,Price,Customer ID
263,489464,21733,85123a mixed,-96,0.0,<NA>
283,489463,71477,short,-240,0.0,<NA>
284,489467,85123A,21733 mixed,-192,0.0,<NA>
470,489521,21646,Unknown Product,-50,0.0,<NA>
3114,489655,20683,Unknown Product,-44,0.0,<NA>
3162,489660,35956,lost,-1043,0.0,<NA>
3168,489663,35605A,damages,-117,0.0,<NA>
4296,489806,18010,Unknown Product,-770,0.0,<NA>
4538,489820,21133,invcd as 84879?,-720,0.0,<NA>
4566,489821,85049G,Unknown Product,-240,0.0,<NA>


In [54]:
special_adjustment_check = df_clean[
    (df_clean["Quantity"] < 0) &
    (~df_clean["IsCancelled"]) &
    (df_clean["Price"] == 0)
]

print("Số dòng cần xem xét:", len(special_adjustment_check))

display(
    special_adjustment_check[
        ["Invoice", "Description", "Quantity", "Price", "Customer ID"]
    ].head(20)
)

Số dòng cần xem xét: 3457


,Invoice,Description,Quantity,Price,Customer ID
263,489464,85123a mixed,-96,0.0,<NA>
283,489463,short,-240,0.0,<NA>
284,489467,21733 mixed,-192,0.0,<NA>
470,489521,Unknown Product,-50,0.0,<NA>
3114,489655,Unknown Product,-44,0.0,<NA>
3162,489660,lost,-1043,0.0,<NA>
3168,489663,damages,-117,0.0,<NA>
4296,489806,Unknown Product,-770,0.0,<NA>
4538,489820,invcd as 84879?,-720,0.0,<NA>
4566,489821,Unknown Product,-240,0.0,<NA>


In [55]:
df_clean.loc[
    (df_clean["Quantity"] < 0) &
    (~df_clean["IsCancelled"]) &
    (df_clean["Price"] == 0),
    "IsSpecialAdjustment"
] = True

print("Số dòng điều chỉnh đặc biệt:")
print(df_clean["IsSpecialAdjustment"].value_counts())

print(
    "Số dòng Quantity âm chưa được đánh dấu:",
    len(df_clean[
        (df_clean["Quantity"] < 0) &
        (~df_clean["IsCancelled"]) &
        (~df_clean["IsSpecialAdjustment"])
    ])
)

Số dòng điều chỉnh đặc biệt:
IsSpecialAdjustment
False    1051781
True        3457
Name: count, dtype: int64
Số dòng Quantity âm chưa được đánh dấu: 0


In [56]:
zero_price = df_clean[df_clean["Price"] == 0]

print("Số dòng Price = 0:", len(zero_price))

print("\nPhân loại theo Quantity:")
print(zero_price["Quantity"].value_counts().head(10))

print("\nPhân loại theo IsCancelled:")
print(zero_price["IsCancelled"].value_counts())

print("\nPhân loại theo IsSpecialAdjustment:")
print(zero_price["IsSpecialAdjustment"].value_counts())

display(
    zero_price[
        ["Invoice", "StockCode", "Description",
         "Quantity", "Price", "Customer ID",
         "IsCancelled", "IsSpecialAdjustment"]
    ].head(30)
)

Số dòng Price = 0: 6191

Phân loại theo Quantity:
Quantity
 1    876
 2    296
-1    212
 3    180
-2    169
-4    107
-3    101
 4     96
 5     91
-5     91
Name: count, dtype: int64

Phân loại theo IsCancelled:
IsCancelled
False    6191
Name: count, dtype: int64

Phân loại theo IsSpecialAdjustment:
IsSpecialAdjustment
True     3457
False    2734
Name: count, dtype: int64


,Invoice,StockCode,Description,Quantity,Price,Customer ID,IsCancelled,IsSpecialAdjustment
263,489464,21733,85123a mixed,-96,0.0,<NA>,False,True
283,489463,71477,short,-240,0.0,<NA>,False,True
284,489467,85123A,21733 mixed,-192,0.0,<NA>,False,True
470,489521,21646,Unknown Product,-50,0.0,<NA>,False,True
3114,489655,20683,Unknown Product,-44,0.0,<NA>,False,True
3161,489659,21350,Unknown Product,230,0.0,<NA>,False,False
3162,489660,35956,lost,-1043,0.0,<NA>,False,True
3168,489663,35605A,damages,-117,0.0,<NA>,False,True
3731,489781,84292,Unknown Product,17,0.0,<NA>,False,False
4296,489806,18010,Unknown Product,-770,0.0,<NA>,False,True


In [57]:
zero_price_remaining = df_clean[
    (df_clean["Price"] == 0) &
    (~df_clean["IsSpecialAdjustment"])
]

print("Số dòng Price = 0 chưa phân loại:", len(zero_price_remaining))

print("\nCác Description phổ biến:")
print(zero_price_remaining["Description"].value_counts().head(30))

print("\nPhân loại Quantity:")
print(zero_price_remaining["Quantity"].value_counts().head(20))

display(
    zero_price_remaining[
        ["Invoice", "StockCode", "Description",
         "Quantity", "Customer ID", "Country"]
    ].head(30)
)

Số dòng Price = 0 chưa phân loại: 2734

Các Description phổ biến:
Description
Unknown Product                       1693
check                                   39
found                                   28
OWL DOORSTOP                            15
adjustment                              14
POLYESTER FILLER PAD 45x45cm            12
amazon                                  11
POLYESTER FILLER PAD 40x40cm            10
IVORY KITCHEN SCALES                    10
FRENCH BLUE METAL DOOR SIGN 1           10
?                                        9
Found                                    9
PICNIC BASKET WICKER LARGE               9
BOX OF 24 COCKTAIL PARASOLS              9
RED KITCHEN SCALES                       9
FRENCH BLUE METAL DOOR SIGN 8            9
CHILDS GARDEN SPADE BLUE                 8
AIRLINE BAG VINTAGE WORLD CHAMPION       8
RECIPE BOX PANTRY YELLOW DESIGN          8
FRENCH BLUE METAL DOOR SIGN 4            8
FRENCH BLUE METAL DOOR SIGN No           8
Manual             

,Invoice,StockCode,Description,Quantity,Customer ID,Country
3161,489659,21350,Unknown Product,230,<NA>,United Kingdom
3731,489781,84292,Unknown Product,17,<NA>,United Kingdom
4674,489825,22076,6 RIBBONS EMPIRE,12,16126,United Kingdom
5904,489861,DOT,DOTCOM POSTAGE,1,<NA>,United Kingdom
6378,489882,35751C,Unknown Product,12,<NA>,United Kingdom
6555,489898,79323G,Unknown Product,954,<NA>,United Kingdom
6581,489903,21166,Unknown Product,48,<NA>,United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,15658,United Kingdom
7204,490015,21982,Unknown Product,467,<NA>,United Kingdom
9249,490123,84508B,Unknown Product,184,<NA>,United Kingdom


In [58]:
print("Phân loại Quantity:")
print(
    zero_price_remaining["Quantity"]
    .apply(lambda x: "Âm" if x < 0 else "Dương" if x > 0 else "Bằng 0")
    .value_counts()
)

print("\nCustomer ID:")
print(zero_price_remaining["Customer ID"].isna().value_counts())

print("\nCác dòng Quantity dương:")
display(
    zero_price_remaining[
        zero_price_remaining["Quantity"] > 0
    ].head(15)
)

print("\nCác dòng Quantity âm:")
display(
    zero_price_remaining[
        zero_price_remaining["Quantity"] < 0
    ].head(15)
)

Phân loại Quantity:
Quantity
Dương    2734
Name: count, dtype: int64

Customer ID:
Customer ID
True     2663
False      71
Name: count, dtype: int64

Các dòng Quantity dương:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancelled,IsSpecialAdjustment,IsNegativePrice
3161,489659,21350,Unknown Product,230,2009-12-01 17:39:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,False,False
3731,489781,84292,Unknown Product,17,2009-12-02 11:45:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,False,False
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126,United Kingdom,Year 2009-2010,False,False,False
5904,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,False,False
6378,489882,35751C,Unknown Product,12,2009-12-02 16:22:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,False,False
6555,489898,79323G,Unknown Product,954,2009-12-03 09:40:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,False,False
6581,489903,21166,Unknown Product,48,2009-12-03 09:57:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,False,False
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658,United Kingdom,Year 2009-2010,False,False,False
7204,490015,21982,Unknown Product,467,2009-12-03 12:29:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,False,False
9249,490123,84508B,Unknown Product,184,2009-12-03 18:08:00,0.0,<NA>,United Kingdom,Year 2009-2010,False,False,False



Các dòng Quantity âm:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancelled,IsSpecialAdjustment,IsNegativePrice


In [59]:
zero_price_with_customer = zero_price_remaining[
    zero_price_remaining["Customer ID"].notna()
]

print("Số dòng:", len(zero_price_with_customer))

display(
    zero_price_with_customer[
        ["Invoice", "StockCode", "Description",
         "Quantity", "Price", "Customer ID"]
    ]
)

Số dòng: 71


,Invoice,StockCode,Description,Quantity,Price,Customer ID
4674,489825,22076,6 RIBBONS EMPIRE,12,0.0,16126
6781,489998,48185,DOOR MAT FAIRY CAKE,2,0.0,15658
16107,490727,M,Manual,1,0.0,17231
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,0.0,14108
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,0.0,14108
...,...,...,...,...,...,...
1004540,577129,22464,HANGING METAL HEART LANTERN,4,0.0,15602
1005014,577168,M,Manual,1,0.0,12603
1006110,577314,23407,SET OF 2 TRAYS HOME SWEET HOME,2,0.0,12444
1011446,577696,M,Manual,1,0.0,16406


In [60]:
numeric_cols = ["Quantity", "Price"]

print("Thống kê mô tả:")
display(df_clean[numeric_cols].describe())

print("\nCác phân vị:")
display(
    df_clean[numeric_cols].quantile(
        [0.25, 0.5, 0.75, 0.95, 0.99]
    )
)

Thống kê mô tả:


,Quantity,Price
count,1.055238e+06,1.055238e+06
mean,1.002209e+01,4.673714e+00
std,1.736819e+02,1.242609e+02
min,-8.099500e+04,-5.359436e+04
25%,1.000000e+00,1.250000e+00
50%,3.000000e+00,2.100000e+00
75%,1.000000e+01,4.150000e+00
max,8.099500e+04,3.897000e+04



Các phân vị:


,Quantity,Price
0.25,1.0,1.25
0.50,3.0,2.10
0.75,10.0,4.15
0.95,30.0,9.95
0.99,101.0,18.68


In [61]:
for col in ["Quantity", "Price"]:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df_clean[
        (df_clean[col] < lower) |
        (df_clean[col] > upper)
    ]

    print(f"\n--- {col} ---")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower bound:", lower)
    print("Upper bound:", upper)
    print("Số lượng outliers:", len(outliers))
    print("Tỷ lệ outliers:", round(len(outliers) / len(df_clean) * 100, 2), "%")


--- Quantity ---
Q1: 1.0
Q3: 10.0
IQR: 9.0
Lower bound: -12.5
Upper bound: 23.5
Số lượng outliers: 116245
Tỷ lệ outliers: 11.02 %

--- Price ---
Q1: 1.25
Q3: 4.15
IQR: 2.9000000000000004
Lower bound: -3.1000000000000005
Upper bound: 8.5
Số lượng outliers: 67789
Tỷ lệ outliers: 6.42 %


In [62]:
# Kiểm tra Quantity
print("QUANTITY LỚN NHẤT:")
display(
    df_clean.nlargest(15, "Quantity")[
        ["Invoice", "StockCode", "Description",
         "Quantity", "Price", "Customer ID",
         "IsCancelled", "IsSpecialAdjustment"]
    ]
)

print("QUANTITY NHỎ NHẤT:")
display(
    df_clean.nsmallest(15, "Quantity")[
        ["Invoice", "StockCode", "Description",
         "Quantity", "Price", "Customer ID",
         "IsCancelled", "IsSpecialAdjustment"]
    ]
)

# Kiểm tra Price
print("PRICE LỚN NHẤT:")
display(
    df_clean.nlargest(15, "Price")[
        ["Invoice", "StockCode", "Description",
         "Quantity", "Price", "Customer ID",
         "IsCancelled", "IsSpecialAdjustment"]
    ]
)

print("PRICE NHỎ NHẤT:")
display(
    df_clean.nsmallest(15, "Price")[
        ["Invoice", "StockCode", "Description",
         "Quantity", "Price", "Customer ID",
         "IsCancelled", "IsSpecialAdjustment"]
    ]
)

QUANTITY LỚN NHẤT:


,Invoice,StockCode,Description,Quantity,Price,Customer ID,IsCancelled,IsSpecialAdjustment
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446,False,False
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346,False,False
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,13902,False,False
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,13902,False,False
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,13902,False,False
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,13902,False,False
1027583,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,0.00,13256,False,False
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,13902,False,False
192197,507637,84016,FLAG OF ST GEORGE CAR FLAG,10200,0.00,<NA>,False,False
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,17940,False,False


QUANTITY NHỎ NHẤT:


,Invoice,StockCode,Description,Quantity,Price,Customer ID,IsCancelled,IsSpecialAdjustment
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,16446,True,False
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,12346,True,False
303996,519017,22759,Unknown Product,-9600,0.00,<NA>,False,True
750990,556690,23005,printing smudges/thrown away,-9600,0.00,<NA>,False,True
750991,556691,23005,printing smudges/thrown away,-9600,0.00,<NA>,False,True
507225,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,0.03,15838,True,False
529729,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,0.03,15838,True,False
156488,504311,22197,Unknown Product,-9200,0.00,<NA>,False,True
750989,556687,23003,Printing smudges/thrown away,-9058,0.00,<NA>,False,True
194372,507913,10120,Zebra invcing error,-9000,0.00,<NA>,False,True


PRICE LỚN NHẤT:


,Invoice,StockCode,Description,Quantity,Price,Customer ID,IsCancelled,IsSpecialAdjustment
748142,C556445,M,Manual,-1,38970.00,15098,True,False
241824,C512770,M,Manual,-1,25111.09,17399,True,False
241827,512771,M,Manual,1,25111.09,<NA>,False,False
320581,C520667,BANK CHARGES,Bank Charges,-1,18910.69,<NA>,True,False
1050063,C580605,AMAZONFEE,AMAZON FEE,-1,17836.46,<NA>,True,False
569163,C540117,AMAZONFEE,AMAZON FEE,-1,16888.02,<NA>,True,False
569164,C540118,AMAZONFEE,AMAZON FEE,-1,16453.71,<NA>,True,False
517953,C537630,AMAZONFEE,AMAZON FEE,-1,13541.33,<NA>,True,False
517955,537632,AMAZONFEE,AMAZON FEE,1,13541.33,<NA>,False,False
519294,C537651,AMAZONFEE,AMAZON FEE,-1,13541.33,<NA>,True,False


PRICE NHỎ NHẤT:


,Invoice,StockCode,Description,Quantity,Price,Customer ID,IsCancelled,IsSpecialAdjustment
179403,A506401,B,Adjust bad debt,1,-53594.36,<NA>,False,False
276274,A516228,B,Adjust bad debt,1,-44031.79,<NA>,False,False
403472,A528059,B,Adjust bad debt,1,-38925.87,<NA>,False,False
825444,A563186,B,Adjust bad debt,1,-11062.06,<NA>,False,False
825445,A563187,B,Adjust bad debt,1,-11062.06,<NA>,False,False
263,489464,21733,85123a mixed,-96,0.00,<NA>,False,True
283,489463,71477,short,-240,0.00,<NA>,False,True
284,489467,85123A,21733 mixed,-192,0.00,<NA>,False,True
470,489521,21646,Unknown Product,-50,0.00,<NA>,False,True
3114,489655,20683,Unknown Product,-44,0.00,<NA>,False,True


In [63]:
df_clean.loc[
    df_clean["Description"].str.contains(
        "Adjust bad debt",
        case=False,
        na=False
    ),
    "IsSpecialAdjustment"
] = True

print(
    "Số dòng điều chỉnh đặc biệt:",
    df_clean["IsSpecialAdjustment"].sum()
)

print(
    "Số dòng Price âm:",
    (df_clean["Price"] < 0).sum()
)

Số dòng điều chỉnh đặc biệt: 3463
Số dòng Price âm: 5


In [64]:
df_clean[
    df_clean["Description"].str.contains(
        "Adjust bad debt",
        case=False,
        na=False
    )
][
    ["Invoice", "StockCode", "Description", "Quantity", "Price",
     "IsSpecialAdjustment"]
]

,Invoice,StockCode,Description,Quantity,Price,IsSpecialAdjustment
179403,A506401,B,Adjust bad debt,1,-53594.36,True
276274,A516228,B,Adjust bad debt,1,-44031.79,True
403472,A528059,B,Adjust bad debt,1,-38925.87,True
825443,A563185,B,Adjust bad debt,1,11062.06,True
825444,A563186,B,Adjust bad debt,1,-11062.06,True
825445,A563187,B,Adjust bad debt,1,-11062.06,True


In [65]:
df_sales_check = df_clean[
    (df_clean["IsCancelled"] == False) &
    (df_clean["IsSpecialAdjustment"] == False) &
    (df_clean["Price"] > 0)
].copy()

print("Kích thước dữ liệu bán hàng:", df_sales_check.shape)

print("\nThống kê Quantity và Price:")
print(df_sales_check[["Quantity", "Price"]].describe())

Kích thước dữ liệu bán hàng: (1029608, 12)

Thống kê Quantity và Price:
           Quantity         Price
count  1.029608e+06  1.029608e+06
mean   1.105796e+01  4.084102e+00
std    1.272406e+02  5.058677e+01
min    1.000000e+00  1.000000e-03
25%    1.000000e+00  1.250000e+00
50%    3.000000e+00  2.100000e+00
75%    1.200000e+01  4.150000e+00
max    8.099500e+04  2.511109e+04


In [66]:
# Tính IQR cho Quantity
Q1_quantity = df_sales_check["Quantity"].quantile(0.25)
Q3_quantity = df_sales_check["Quantity"].quantile(0.75)
IQR_quantity = Q3_quantity - Q1_quantity

upper_quantity = Q3_quantity + 1.5 * IQR_quantity

# Tính IQR cho Price
Q1_price = df_sales_check["Price"].quantile(0.25)
Q3_price = df_sales_check["Price"].quantile(0.75)
IQR_price = Q3_price - Q1_price

upper_price = Q3_price + 1.5 * IQR_price

# Đếm outliers
outliers_quantity = df_sales_check[
    df_sales_check["Quantity"] > upper_quantity
]

outliers_price = df_sales_check[
    df_sales_check["Price"] > upper_price
]

print("Quantity:")
print("Ngưỡng trên:", upper_quantity)
print("Số outliers:", len(outliers_quantity))

print("\nPrice:")
print("Ngưỡng trên:", upper_price)
print("Số outliers:", len(outliers_price))

Quantity:
Ngưỡng trên: 28.5
Số outliers: 53926

Price:
Ngưỡng trên: 8.5
Số outliers: 64671


In [67]:
outliers_quantity.nlargest(
    10, "Quantity"
)[
    ["Invoice", "StockCode", "Description",
     "Quantity", "Price", "Customer ID", "Country"]
]

,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446,United Kingdom
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346,United Kingdom
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,13902,Denmark
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,13902,Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,13902,Denmark
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,13902,Denmark
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,13902,Denmark
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,17940,United Kingdom
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,17940,United Kingdom
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,17940,United Kingdom


In [68]:
outliers_price.nlargest(
    10, "Price"
)[
    ["Invoice", "StockCode", "Description",
     "Quantity", "Price", "Customer ID", "Country"]
]

,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
241827,512771,M,Manual,1,25111.09,<NA>,United Kingdom
517955,537632,AMAZONFEE,AMAZON FEE,1,13541.33,<NA>,United Kingdom
540478,537632,AMAZONFEE,AMAZON FEE,1,13541.33,<NA>,United Kingdom
135013,502263,M,Manual,1,10953.50,12918,United Kingdom
135015,502265,M,Manual,1,10953.50,<NA>,United Kingdom
342147,522796,M,Manual,1,10468.80,<NA>,United Kingdom
358639,524159,M,Manual,1,10468.80,14063,United Kingdom
372834,525399,M,Manual,1,10468.80,<NA>,United Kingdom
74356,496115,M,Manual,1,8985.60,17949,United Kingdom
698843,551697,POST,POSTAGE,1,8142.75,16029,United Kingdom


In [69]:
# Thống kê các StockCode trong nhóm Price outliers
print(
    outliers_price["StockCode"]
    .value_counts()
    .head(20)
)

StockCode
22423     4036
POST      1827
21843     1553
DOT       1412
21217      958
85066      950
22487      788
22424      752
22501      732
22776      665
22605      652
22622      569
21258      561
37503      521
82486      508
21623      505
22847      493
72760B     470
22061      469
22635      455
Name: count, dtype: Int64


In [70]:
special_codes = ["M", "POST", "DOT", "AMAZONFEE"]

for code in special_codes:
    print(f"\nStockCode: {code}")

    display(
        df_sales_check[
            df_sales_check["StockCode"] == code
        ][
            ["Invoice", "StockCode", "Description",
             "Quantity", "Price", "Customer ID"]
        ].head(5)
    )


StockCode: M


,Invoice,StockCode,Description,Quantity,Price,Customer ID
2697,489609,M,Manual,1,4.00,<NA>
11310,490300,M,Manual,1,0.85,12970
11311,490300,M,Manual,1,0.21,12970
17386,490760,M,Manual,1,10.00,14295
17887,490881,M,Manual,1,10.00,16210



StockCode: POST


,Invoice,StockCode,Description,Quantity,Price,Customer ID
89,489439,POST,POSTAGE,3,18.0,12682
126,489444,POST,POSTAGE,1,141.0,12636
173,489447,POST,POSTAGE,1,130.0,12362
625,489526,POST,POSTAGE,6,18.0,12533
1244,489557,POST,POSTAGE,4,18.0,12490



StockCode: DOT


,Invoice,StockCode,Description,Quantity,Price,Customer ID
2379,489597,DOT,DOTCOM POSTAGE,1,647.19,<NA>
2539,489600,DOT,DOTCOM POSTAGE,1,55.96,<NA>
2551,489601,DOT,DOTCOM POSTAGE,1,68.39,<NA>
2571,489602,DOT,DOTCOM POSTAGE,1,59.35,<NA>
2619,489603,DOT,DOTCOM POSTAGE,1,42.39,<NA>



StockCode: AMAZONFEE


,Invoice,StockCode,Description,Quantity,Price,Customer ID
440698,531411,AMAZONFEE,AMAZON FEE,1,6706.71,<NA>
517955,537632,AMAZONFEE,AMAZON FEE,1,13541.33,<NA>
540478,537632,AMAZONFEE,AMAZON FEE,1,13541.33,<NA>
660995,547901,AMAZONFEE,AMAZON FEE,1,219.76,<NA>


In [71]:
special_codes = ["M", "POST", "DOT", "AMAZONFEE"]

special_summary = (
    df_sales_check[
        df_sales_check["StockCode"].isin(special_codes)
    ]
    .groupby(["StockCode", "Description"])
    .agg(
        Total_Rows=("Invoice", "count"),
        Total_Quantity=("Quantity", "sum"),
        Total_Revenue=("Price", "sum")
    )
    .reset_index()
)

display(special_summary)

,StockCode,Description,Total_Rows,Total_Quantity,Total_Revenue
0,AMAZONFEE,AMAZON FEE,4,4,34009.13
1,DOT,DOTCOM POSTAGE,1436,1436,322657.48
2,M,Manual,866,9748,334606.99
3,POST,POSTAGE,1886,5461,55241.52


In [72]:
special_codes = ["M", "POST", "DOT", "AMAZONFEE"]

df_product_sales = df_sales_check[
    ~df_sales_check["StockCode"].isin(special_codes)
].copy()

print("Dữ liệu bán hàng ban đầu:", df_sales_check.shape)
print("Dữ liệu bán sản phẩm:", df_product_sales.shape)

print("\nKiểm tra các mã đặc biệt còn lại:")
print(
    df_product_sales[
        df_product_sales["StockCode"].isin(special_codes)
    ].shape
)

Dữ liệu bán hàng ban đầu: (1029608, 12)
Dữ liệu bán sản phẩm: (1025416, 12)

Kiểm tra các mã đặc biệt còn lại:
(0, 12)


In [73]:
# Quantity
Q1_quantity = df_product_sales["Quantity"].quantile(0.25)
Q3_quantity = df_product_sales["Quantity"].quantile(0.75)
IQR_quantity = Q3_quantity - Q1_quantity
upper_quantity = Q3_quantity + 1.5 * IQR_quantity

# Price
Q1_price = df_product_sales["Price"].quantile(0.25)
Q3_price = df_product_sales["Price"].quantile(0.75)
IQR_price = Q3_price - Q1_price
upper_price = Q3_price + 1.5 * IQR_price

# Đếm outliers
outliers_quantity_product = df_product_sales[
    df_product_sales["Quantity"] > upper_quantity
]

outliers_price_product = df_product_sales[
    df_product_sales["Price"] > upper_price
]

print("QUANTITY")
print("Ngưỡng trên:", upper_quantity)
print("Số outliers:", len(outliers_quantity_product))

print("\nPRICE")
print("Ngưỡng trên:", upper_price)
print("Số outliers:", len(outliers_price_product))

QUANTITY
Ngưỡng trên: 28.5
Số outliers: 53884

PRICE
Ngưỡng trên: 8.45
Số outliers: 77609


In [74]:
outliers_price_product.nlargest(
    10, "Price"
)[
    ["Invoice", "StockCode", "Description",
     "Quantity", "Price", "Customer ID", "Country"]
]

,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
71077,495798,ADJUST,Adjustment by john on 26/01/2010 17,1,5117.03,<NA>,United Kingdom
192196,507637,84016,FLAG OF ST GEORGE CAR FLAG,1,1157.15,<NA>,United Kingdom
136403,502451,84016,FLAG OF ST GEORGE CAR FLAG,1,867.79,<NA>,United Kingdom
748132,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,649.50,15098,United Kingdom
748143,556446,22502,PICNIC BASKET WICKER 60 PIECES,1,649.50,15098,United Kingdom
180997,506571,84016,FLAG OF ST GEORGE CAR FLAG,1,408.40,<NA>,United Kingdom
71082,495802,ADJUST,Adjustment by john on 26/01/2010 17,1,387.54,12748,United Kingdom
249673,513566,ADJUST2,Adjustment by Peter on Jun 25 2010,1,358.47,17364,United Kingdom
71038,495755,ADJUST,Adjustment by john on 26/01/2010 16,1,342.80,14413,United Kingdom
249672,513564,ADJUST2,Adjustment by Peter on Jun 25 2010,1,300.13,17050,United Kingdom


In [75]:
adjustment_codes = ["ADJUST", "ADJUST2"]

adjustment_summary = (
    df_product_sales[
        df_product_sales["StockCode"].isin(adjustment_codes)
    ]
    .groupby(["StockCode", "Description"])
    .agg(
        Total_Rows=("Invoice", "count"),
        Total_Quantity=("Quantity", "sum"),
        Min_Price=("Price", "min"),
        Max_Price=("Price", "max")
    )
    .reset_index()
)

display(adjustment_summary)

,StockCode,Description,Total_Rows,Total_Quantity,Min_Price,Max_Price
0,ADJUST,Adjustment by john on 26/01/2010 16,20,20,4.57,342.80
1,ADJUST,Adjustment by john on 26/01/2010 17,16,16,11.31,5117.03
2,ADJUST2,Adjustment by Peter on Jun 25 2010,3,3,72.45,358.47


In [76]:
adjustment_codes = ["ADJUST", "ADJUST2"]

df_clean.loc[
    df_clean["StockCode"].isin(adjustment_codes),
    "IsSpecialAdjustment"
] = True

print("Tổng số dòng điều chỉnh đặc biệt:")
print(df_clean["IsSpecialAdjustment"].sum())

print("\nSố dòng ADJUST và ADJUST2:")
print(
    df_clean["StockCode"]
    .isin(adjustment_codes)
    .sum()
)

Tổng số dòng điều chỉnh đặc biệt:
3533

Số dòng ADJUST và ADJUST2:
70


In [77]:
adjustment_codes = ["ADJUST", "ADJUST2"]

adjustment_check = df_clean[
    df_clean["StockCode"].isin(adjustment_codes)
]

print("Phân loại theo trạng thái:")
print(
    adjustment_check.groupby(
        ["StockCode", "IsCancelled", "IsSpecialAdjustment"]
    ).size()
)

print("\nDanh sách mô tả:")
print(adjustment_check["Description"].value_counts())

Phân loại theo trạng thái:
StockCode  IsCancelled  IsSpecialAdjustment
ADJUST     False        True                   36
           True         True                   31
ADJUST2    False        True                    3
dtype: int64

Danh sách mô tả:
Description
Adjustment by john on 26/01/2010 16    38
Adjustment by john on 26/01/2010 17    26
Adjustment by Peter on 24/05/2010 1     3
Adjustment by Peter on Jun 25 2010      3
Name: count, dtype: Int64


In [78]:
df_sales_check = df_clean[
    (~df_clean["IsCancelled"]) &
    (~df_clean["IsSpecialAdjustment"]) &
    (df_clean["Price"] > 0)
].copy()

special_codes = ["M", "POST", "DOT", "AMAZONFEE"]

df_product_sales = df_sales_check[
    ~df_sales_check["StockCode"].isin(special_codes)
].copy()

print("df_clean:", df_clean.shape)
print("df_sales_check:", df_sales_check.shape)
print("df_product_sales:", df_product_sales.shape)

df_clean: (1055238, 12)
df_sales_check: (1029569, 12)
df_product_sales: (1025377, 12)


In [79]:
numeric_cols = ["Quantity", "Price"]

for col in numeric_cols:
    Q1 = df_product_sales[col].quantile(0.25)
    Q3 = df_product_sales[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_product_sales[
        (df_product_sales[col] < lower_bound) |
        (df_product_sales[col] > upper_bound)
    ]

    print(f"\n--- {col} ---")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)
    print("Số lượng ngoại lệ:", len(outliers))
    print("Tỷ lệ ngoại lệ:", round(len(outliers) / len(df_product_sales) * 100, 2), "%")


--- Quantity ---
Q1: 1.0
Q3: 12.0
IQR: 11.0
Lower bound: -15.5
Upper bound: 28.5
Số lượng ngoại lệ: 53884
Tỷ lệ ngoại lệ: 5.26 %

--- Price ---
Q1: 1.25
Q3: 4.13
IQR: 2.88
Lower bound: -3.0700000000000003
Upper bound: 8.45
Số lượng ngoại lệ: 77571
Tỷ lệ ngoại lệ: 7.57 %


In [80]:
high_price = df_product_sales.nlargest(10, "Price")

print(
    high_price[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "Customer ID",
            "Country"
        ]
    ].to_string(index=False)
)

Invoice StockCode                    Description  Quantity   Price  Customer ID        Country
 507637     84016     FLAG OF ST GEORGE CAR FLAG         1 1157.15         <NA> United Kingdom
 502451     84016     FLAG OF ST GEORGE CAR FLAG         1  867.79         <NA> United Kingdom
 556444     22502 PICNIC BASKET WICKER 60 PIECES        60  649.50        15098 United Kingdom
 556446     22502 PICNIC BASKET WICKER 60 PIECES         1  649.50        15098 United Kingdom
 506571     84016     FLAG OF ST GEORGE CAR FLAG         1  408.40         <NA> United Kingdom
 515349     22656   VINTAGE BLUE KITCHEN CABINET         1  295.00        15513 United Kingdom
 515349     22655    VINTAGE RED KITCHEN CABINET         1  295.00        15513 United Kingdom
 515403     22655    VINTAGE RED KITCHEN CABINET         1  295.00         <NA> United Kingdom
 515602     22656   VINTAGE BLUE KITCHEN CABINET         1  295.00        14895 United Kingdom
 515811     22656   VINTAGE BLUE KITCHEN CABINET  

In [81]:
high_quantity = df_product_sales.nlargest(10, "Quantity")

print(
    high_quantity[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "Customer ID",
            "Country"
        ]
    ].to_string(index=False)
)

Invoice StockCode                        Description  Quantity  Price  Customer ID        Country
 581483     23843        PAPER CRAFT , LITTLE BIRDIE     80995   2.08        16446 United Kingdom
 541431     23166     MEDIUM CERAMIC TOP STORAGE JAR     74215   1.04        12346 United Kingdom
 497946     37410 BLACK AND WHITE PAISLEY FLOWER MUG     19152   0.10        13902        Denmark
 501534     21099        SET/6 STRAWBERRY PAPER CUPS     12960   0.10        13902        Denmark
 501534     21091        SET/6 WOODLAND PAPER PLATES     12960   0.10        13902        Denmark
 501534     21085          SET/6 WOODLAND PAPER CUPS     12744   0.10        13902        Denmark
 501534     21092      SET/6 STRAWBERRY PAPER PLATES     12480   0.10        13902        Denmark
 502269     21984    PACK OF 12 PINK PAISLEY TISSUES     10000   0.25        17940 United Kingdom
 502269     21982            PACK OF 12 SUKI TISSUES     10000   0.25        17940 United Kingdom
 502269     21980   

In [82]:
print("Kích thước df_product_sales:", df_product_sales.shape)

print("\nSố lượng giá trị thiếu:")
print(df_product_sales.isnull().sum())

print("\nTổng số dòng trùng lặp:")
print(df_product_sales.duplicated().sum())

Kích thước df_product_sales: (1025377, 12)

Số lượng giá trị thiếu:
Invoice                     0
StockCode                   0
Description                 0
Quantity                    0
InvoiceDate                 0
Price                       0
Customer ID            234356
Country                     0
SourceSheet                 0
IsCancelled                 0
IsSpecialAdjustment         0
IsNegativePrice             0
dtype: int64

Tổng số dòng trùng lặp:
0


In [83]:
print("Kiểm tra Quantity:")
print("Âm:", (df_product_sales["Quantity"] < 0).sum())
print("Bằng 0:", (df_product_sales["Quantity"] == 0).sum())

print("\nKiểm tra Price:")
print("Âm:", (df_product_sales["Price"] < 0).sum())
print("Bằng 0:", (df_product_sales["Price"] == 0).sum())

Kiểm tra Quantity:
Âm: 0
Bằng 0: 0

Kiểm tra Price:
Âm: 0
Bằng 0: 0


In [84]:
special_codes = [
    "M",
    "POST",
    "DOT",
    "AMAZONFEE",
    "ADJUST",
    "ADJUST2",
    "B"
]

for code in special_codes:
    count = (df_product_sales["StockCode"] == code).sum()
    print(f"{code}: {count} dòng")
    

M: 0 dòng
POST: 0 dòng
DOT: 0 dòng
AMAZONFEE: 0 dòng
ADJUST: 0 dòng
ADJUST2: 0 dòng
B: 0 dòng


In [85]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Xuất dữ liệu đã làm sạch
df_clean.to_csv(
    output_dir / "online_retail_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

# Xuất dữ liệu bán sản phẩm
df_product_sales.to_csv(
    output_dir / "online_retail_product_sales.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất dữ liệu thành công!")
print("Cleaned:", df_clean.shape)
print("Product sales:", df_product_sales.shape)

Đã xuất dữ liệu thành công!
Cleaned: (1055238, 12)
Product sales: (1025377, 12)


In [86]:
# Đọc lại hai file CSV
df_clean_check = pd.read_csv(
    "../data/processed/online_retail_cleaned.csv"
)

df_product_check = pd.read_csv(
    "../data/processed/online_retail_product_sales.csv"
)

print("Cleaned CSV:", df_clean_check.shape)
print("Product sales CSV:", df_product_check.shape)

print("\n5 dòng đầu của Product Sales:")
display(df_product_check.head())

print("\nKiểm tra dữ liệu thiếu:")
print(df_product_check.isnull().sum())

Cleaned CSV: (1055238, 12)
Product sales CSV: (1025377, 12)

5 dòng đầu của Product Sales:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancelled,IsSpecialAdjustment,IsNegativePrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010,False,False,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,False,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,False,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010,False,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010,False,False,False



Kiểm tra dữ liệu thiếu:
Invoice                     0
StockCode                   0
Description                 0
Quantity                    0
InvoiceDate                 0
Price                       0
Customer ID            234356
Country                     0
SourceSheet                 0
IsCancelled                 0
IsSpecialAdjustment         0
IsNegativePrice             0
dtype: int64
